<a href="https://colab.research.google.com/github/saanidhi-git/sae-feature-similarity/blob/main/notebooks/train_sae_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_DIR = "/content/drive/MyDrive/sae-feature-similarity"

os.makedirs(PROJECT_DIR, exist_ok=True)

print(PROJECT_DIR)

/content/drive/MyDrive/sae-feature-similarity


In [3]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [4]:
!pip install -q torch
!pip install -q transformer-lens
!pip install -q sae-lens
!pip install -q scikit-learn
!pip install -q matplotlib

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.7/977.7 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 84.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.0/311.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 17.2 MB/s eta 0:00:00


In [5]:
import torch
import transformer_lens
import sae_lens
import sklearn

print("Torch:", torch.__version__)
print("SAELens:", sae_lens.__version__)
print("READY")

Torch: 2.11.0+cu128
SAELens: 6.44.2
READY


In [6]:
from sae_lens import LanguageModelSAERunnerConfig
import inspect

print(inspect.signature(LanguageModelSAERunnerConfig))

(sae: +T_TRAINING_SAE_CONFIG, model_name: str = 'gelu-2l', model_class_name: str = 'HookedTransformer', hook_name: str = 'blocks.0.hook_mlp_out', hook_eval: str = 'NOT_IN_USE', hook_head_index: int | None = None, dataset_path: str = '', dataset_trust_remote_code: bool = True, streaming: bool = True, is_dataset_tokenized: bool = True, use_chat_formatting: bool = False, context_size: int = 128, use_cached_activations: bool = False, cached_activations_path: str | None = None, from_pretrained_path: str | None = None, n_batches_in_buffer: int = 20, training_tokens: int = 2000000, store_batch_size_prompts: int = 32, seqpos_slice: tuple[int | None, ...] = (None,), disable_concat_sequences: bool = False, sequence_separator_token: Union[int, Literal['bos', 'eos', 'sep'], NoneType] = 'bos', activations_mixing_fraction: float = 0.5, device: str = 'cpu', llm_device: str | None = None, act_store_device: str | None = None, prefetch_llm_batches: bool | int = False, seed: int = 42, dtype: str = 'float

In [7]:
import sae_lens

print(dir(sae_lens))

['ActivationsStore', 'BatchTopKTrainingSAE', 'BatchTopKTrainingSAEConfig', 'CacheActivationsRunner', 'CacheActivationsRunnerConfig', 'GatedSAE', 'GatedSAEConfig', 'GatedTrainingSAE', 'GatedTrainingSAEConfig', 'HookedSAETransformer', 'JumpReLUSAE', 'JumpReLUSAEConfig', 'JumpReLUSkipTranscoder', 'JumpReLUSkipTranscoderConfig', 'JumpReLUTrainingSAE', 'JumpReLUTrainingSAEConfig', 'JumpReLUTranscoder', 'JumpReLUTranscoderConfig', 'LanguageModelSAERunnerConfig', 'LanguageModelSAETrainingRunner', 'LoggingConfig', 'MatchingPursuitSAE', 'MatchingPursuitSAEConfig', 'MatchingPursuitTrainingSAE', 'MatchingPursuitTrainingSAEConfig', 'MatryoshkaBatchTopKTrainingSAE', 'MatryoshkaBatchTopKTrainingSAEConfig', 'MultiSAEEvaluator', 'MultiSAETrainingRunner', 'MultiSAETrainingRunnerConfig', 'PretokenizeRunner', 'PretokenizeRunnerConfig', 'PretrainedSaeDiskLoader', 'PretrainedSaeHuggingfaceLoader', 'SAE', 'SAEConfig', 'SAETrainer', 'SAETrainingRunner', 'SAETransformerBridge', 'SkipTranscoder', 'SkipTranscod

In [8]:
from sae_lens import StandardTrainingSAEConfig
import inspect

print(inspect.signature(StandardTrainingSAEConfig))

(d_in: int, d_sae: int, dtype: str = 'float32', device: str = 'cpu', apply_b_dec_to_input: bool = True, normalize_activations: Literal['none', 'expected_average_only_in', 'layer_norm'] = 'none', reshape_activations: Literal['none', 'hook_z'] = 'none', metadata: sae_lens.saes.sae.SAEMetadata = <factory>, l1_coefficient: float = 1.0, lp_norm: float = 1.0, l1_warm_up_steps: int = 0, *, decoder_init_norm: float | None = 0.1) -> None


In [9]:
from sae_lens import TrainingSAEConfig
import inspect

print(inspect.signature(TrainingSAEConfig))

(d_in: int, d_sae: int, dtype: str = 'float32', device: str = 'cpu', apply_b_dec_to_input: bool = True, normalize_activations: Literal['none', 'expected_average_only_in', 'layer_norm'] = 'none', reshape_activations: Literal['none', 'hook_z'] = 'none', metadata: sae_lens.saes.sae.SAEMetadata = <factory>, *, decoder_init_norm: float | None = 0.1) -> None


In [10]:
from sae_lens import LanguageModelSAETrainingRunner
import inspect

print(inspect.signature(LanguageModelSAETrainingRunner))

(cfg: sae_lens.config.LanguageModelSAERunnerConfig[~T_TRAINING_SAE_CONFIG], override_dataset: datasets.dataset_dict.DatasetDict | datasets.arrow_dataset.Dataset | datasets.dataset_dict.IterableDatasetDict | datasets.iterable_dataset.IterableDataset | None = None, override_model: transformer_lens.HookedRootModule.HookedRootModule | None = None, override_sae: Optional[sae_lens.saes.sae.TrainingSAE[Any]] = None, resume_from_checkpoint: pathlib.Path | str | None = None)


In [11]:
from sae_lens import SAETrainingRunner
import inspect

print(inspect.signature(SAETrainingRunner))

(*args, **kwargs)


In [12]:
from sae_lens import (
    StandardTrainingSAEConfig,
    LanguageModelSAERunnerConfig
)

sae_cfg = StandardTrainingSAEConfig(
    d_in=768,      # GPT-2 residual stream width
    d_sae=3072,    # 4x expansion
    device="cuda",
    dtype="float32",
    l1_coefficient=1e-3
)

print(sae_cfg)

StandardTrainingSAEConfig(d_in=768, d_sae=3072, dtype='float32', device='cuda', apply_b_dec_to_input=True, normalize_activations='none', reshape_activations='none', metadata=SAEMetadata({'sae_lens_version': '6.44.2', 'sae_lens_training_version': '6.44.2'}), decoder_init_norm=0.1, l1_coefficient=0.001, lp_norm=1.0, l1_warm_up_steps=0)


In [13]:
from sae_lens import (
    StandardTrainingSAEConfig,
    LanguageModelSAERunnerConfig
)

sae_cfg = StandardTrainingSAEConfig(
    d_in=768,
    d_sae=3072,
    device="cuda",
    dtype="float32",
    l1_coefficient=1e-3
)

runner_cfg = LanguageModelSAERunnerConfig(
    sae=sae_cfg,

    model_name="gpt2",

    hook_name="blocks.6.hook_resid_pre",

    dataset_path="roneneldan/TinyStories",

    streaming=True,

    context_size=128,

    training_tokens=50_000,

    train_batch_size_tokens=4096,

    device="cuda",

    seed=42,

    n_checkpoints=0,

    verbose=True
)

print(runner_cfg)

LanguageModelSAERunnerConfig(sae=StandardTrainingSAEConfig(d_in=768, d_sae=3072, dtype='float32', device='cuda', apply_b_dec_to_input=True, normalize_activations='none', reshape_activations='none', metadata=SAEMetadata({'sae_lens_version': '6.44.2', 'sae_lens_training_version': '6.44.2'}), decoder_init_norm=0.1, l1_coefficient=0.001, lp_norm=1.0, l1_warm_up_steps=0), model_name='gpt2', model_class_name='HookedTransformer', hook_name='blocks.6.hook_resid_pre', hook_eval='NOT_IN_USE', hook_head_index=None, dataset_path='roneneldan/TinyStories', dataset_trust_remote_code=True, streaming=True, is_dataset_tokenized=True, use_chat_formatting=False, context_size=128, use_cached_activations=False, cached_activations_path=None, from_pretrained_path=None, n_batches_in_buffer=20, training_tokens=50000, store_batch_size_prompts=32, seqpos_slice=(None,), disable_concat_sequences=False, sequence_separator_token='bos', activations_mixing_fraction=0.5, device='cuda', llm_device='cuda', act_store_devic

In [14]:
from sae_lens import LanguageModelSAETrainingRunner

runner = LanguageModelSAETrainingRunner(runner_cfg)

print("Runner Created Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'roneneldan/TinyStories' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'roneneldan/TinyStories' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loaded pretrained model gpt2 into HookedTransformer


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

Runner Created Successfully


/usr/local/lib/python3.12/dist-packages/sae_lens/training/activations_store.py:455: UserWarning: Dataset is not tokenized. Pre-tokenizing will improve performance and allows for more control over special tokens. See https://decoderesearch.github.io/SAELens/training_saes/#pretokenizing-datasets for more info.
  warnings.warn(


In [25]:
result = runner.run()

Training SAE:   0%|          | 0/10000 [00:00<?, ?it/s]

In [17]:
is_dataset_tokenized=False

from sae_lens import LoggingConfig

logger = LoggingConfig(
    log_to_wandb=False
)

dataset_path="roneneldan/TinyStories"
is_dataset_tokenized=False
logger=logger

In [18]:
from datasets import load_dataset

ds = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
)

sample = next(iter(ds))
print(sample)

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [19]:
from sae_lens import (
    StandardTrainingSAEConfig,
    LanguageModelSAERunnerConfig,
    LoggingConfig
)

logger = LoggingConfig(
    log_to_wandb=False
)

sae_cfg = StandardTrainingSAEConfig(
    d_in=768,
    d_sae=3072,
    device="cuda",
    dtype="float32",
    l1_coefficient=1e-3
)

runner_cfg = LanguageModelSAERunnerConfig(
    sae=sae_cfg,

    model_name="gpt2",

    hook_name="blocks.6.hook_resid_pre",

    dataset_path="roneneldan/TinyStories",

    streaming=True,

    is_dataset_tokenized=False,

    context_size=128,

    training_tokens=10000,

    train_batch_size_tokens=2048,

    device="cuda",

    seed=42,

    logger=logger,

    n_checkpoints=0,

    verbose=True
)

print("CONFIG READY")

CONFIG READY


In [23]:
from sae_lens import LanguageModelSAETrainingRunner

runner = LanguageModelSAETrainingRunner(runner_cfg)

print("RUNNER READY")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'roneneldan/TinyStories' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'roneneldan/TinyStories' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loaded pretrained model gpt2 into HookedTransformer
RUNNER READY


In [26]:
result = runner.run()

Training SAE:   0%|          | 0/10000 [00:00<?, ?it/s]

In [27]:
import torch

print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

1.9810190200805664 GB allocated
2.54296875 GB reserved


In [28]:
!nvidia-smi

Thu Jun  4 16:25:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             32W /   70W |    2743MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----